# Compare ViT with Swin, CoAtNet, DeiT

                                                                                                                                                    
- Swin Transformer: https://arxiv.org/abs/2103.14030                                                                                                 
- DeiT (Data-efficient Image Transformers): https://arxiv.org/abs/2012.12877                                                                         
- CoAtNet: https://arxiv.org/abs/2106.04803                                                                                                          
- Attention Is All You Need: https://arxiv.org/abs/1706.03762
- timm repo (has all these models implemented): https://github.com/huggingface/pytorch-image-models
- The Annotated Transformer (Harvard NLP walkthrough): https://nlp.seas.harvard.edu/annotated-transformer/

### Key Ideas Oversimplified:

#### Swin Transformer

* **Shifted windows**. Instead of global attention (every patch attends to every patch), Swin gets attentoin within local windows. (for example 7x7 patches). Then, it shifts the window grid by half a window between layers do information can flow across boundaries. 
* **Hierarchichal feature maps.** unlike ViT which keeps the same resolution thrughout, Swin progressively merges patches. (Think of this as pooling in CNNs). This gives us multi-scale features which will become very important in detection and segmentation later on. 
* Overall, Swim gives efficiency and versatiltiy. You can now get comparable accuracy with local attention of you shift the windows in a clever way.


#### DEiT

* showed you can get competitive results with less data, as long as you have the right training recipe. String augmentation, good regularization, and some careful hyperparameter tuning. 
* **Distillation token**. DEiT will add a second special token on top of the cls_token to learn from a CNN model. The CNN can teach the ViT what to attend to during training.

#### CoAtNet

* Brings in the best of both worlds. CNNS are good at local geatures and have nice inductive biases. Transformers are good at global relationships. If you stack CNNs and transformer blocks, you get good results. 
* **Relative Attention**. Take away the absolute positional embeddings like we had in the vanilla ViT, and instead use relative position biases in attentnions scores. This generalizes nicely to different input sizes. 


Building on top of our ViT, we export everything we wrote in the first module to vit.py. We can import that work and build on top of it

In [2]:
from vit import ViT, PatchEmbedding, MultiHeadAttention, TransformerEncoder                                                                          

## Baseline Vanilla ViT on CIFAR-100

In [9]:
from torchvision.datasets import CIFAR100
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import torch.nn.functional as F # This gives us the softmax()
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((56, 56)),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

train_dataset = CIFAR100(root='./data', train=True, download=True, transform=transform)
test_dataset = CIFAR100(root='./data', train=False, download=True, transform=transform)

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using: {device}")

Using: mps


In [11]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [12]:
model = ViT(
      img_size=56, patch_size=7, num_hiddens=256,
      num_heads=8, num_classes=100, depth=2, dropout=0.1
  ).to(device)

In [13]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(dim=-1) == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Acc: {correct/total:.4f}")

Epoch 1 | Loss: 3.6897 | Acc: 0.1349
Epoch 2 | Loss: 3.1289 | Acc: 0.2273
Epoch 3 | Loss: 2.8855 | Acc: 0.2782
Epoch 4 | Loss: 2.7187 | Acc: 0.3108
Epoch 5 | Loss: 2.5687 | Acc: 0.3399
